In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

!unzip -q drive/MyDrive/dataset.zip

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as pl
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.models import Model
from pathlib import Path

In [ ]:
# ----------------------------------------
# PATHS & LABELS
# ----------------------------------------
TRAIN_PATH = Path('dataset/training')
VAL_PATH = Path('dataset/validation')
LABELS = ['intact', 'destroyed']
CLASS_TO_INT = {'intact': 0, 'destroyed': 1}
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

In [ ]:
# ----------------------------------------
# PLOT CLASS DISTRIBUTION
# ----------------------------------------
counts = {}
for l in LABELS:
    counts[l] = len(os.listdir(TRAIN_PATH / l))
    print(l, counts[l])

pl.figure(figsize=(5, 5))
pl.bar(range(len(counts)), list(counts.values()), align='center')
pl.xticks(range(len(counts)), list(counts.keys()), fontsize=12, rotation=40)
pl.xlabel('Class Label', fontsize=13)
pl.ylabel('Class Size', fontsize=13)
pl.title('Class Distribution', fontsize=15)
pl.show()

In [ ]:
# ----------------------------------------
# DATASET LOADING
# ----------------------------------------
def load_image(path):
    image = tf.io.read_file(path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    return image

def get_label_from_path_py(filepath):
    # Convert to string if it's a Tensor
    filepath = filepath.numpy().decode('utf-8') if isinstance(filepath, tf.Tensor) else str(filepath)
    label_str = Path(filepath).parent.name
    return np.int32(CLASS_TO_INT[label_str])

def decode_and_label(file_path):
    image = load_image(file_path)
    label = tf.py_function(func=get_label_from_path_py, inp=[file_path], Tout=tf.int32)
    label.set_shape(())
    return image, label

def get_dataset(directory):
    all_image_paths = tf.data.Dataset.list_files(str(directory / '*' / '*.png'), shuffle=True)
    dataset = all_image_paths.map(decode_and_label, num_parallel_calls=AUTOTUNE)
    return dataset

train_ds_raw = get_dataset(TRAIN_PATH)
val_ds_raw = get_dataset(VAL_PATH)

In [ ]:
# ----------------------------------------
# PLOT RAW TRAIN IMAGES
# ----------------------------------------
N_IMAGES = 20
fig, axs = pl.subplots(4, 5, figsize=(12, 8))
axs = axs.ravel()
for idx, (image_raw, label) in enumerate(train_ds_raw.take(N_IMAGES)):
    axs[idx].imshow(image_raw.numpy().astype("uint8"))
    axs[idx].axis('off')
    axs[idx].set_title(str(label.numpy()))
pl.tight_layout()
pl.show()

In [ ]:
# ----------------------------------------
# PREPROCESS FOR VGG16
# ----------------------------------------
def preprocess_vgg(image, label):
    image = tf.keras.applications.vgg16.preprocess_input(image)
    return image, label

train_ds = train_ds_raw.map(preprocess_vgg, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = val_ds_raw.map(preprocess_vgg, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

## TODO

Implement the create_model() function to build a binary classification model using VGG16 from Keras, pretrained on ImageNet. https://keras.io/api/applications/vgg/vgg_models/#vgg16-function

The model should:
- Use VGG16 with ImageNet pretrained weights, without its original classification head.
- Use global average pooling to obtain the feature representation.
- Freeze the first freeze_n_first_layers layers of the VGG16 base model.
- Add a classification head consisting of two fully connected layers with 64 neurons and ReLU activation, each preceded by a Dropout layer with a rate of 0.1, followed by a final output layer with n_outputs neurons and sigmoid activation.
- Return the model.

Display the model architecture using model.summary().
The final model should therefore combine a pretrained VGG16 feature extractor with a custom classification head adapted to the task.

In [ ]:
def create_model(input_shape, n_outputs, freeze_n_first_layers=14):
    ...

In [ ]:
# ----------------------------------------
# COMPILE MODEL
# ----------------------------------------
model.compile(
    optimizer=tf.keras.optimizers.SGD(
        learning_rate=1e-4,
        weight_decay=1e-4,
        momentum=0.99,
        clipnorm=1
    ),
    loss='binary_crossentropy',
    metrics=['binary_accuracy']
)

In [ ]:

# ----------------------------------------
# TRAIN MODEL
# ----------------------------------------
history = model.fit(
    train_ds,
    epochs=5,
    validation_data=val_ds
)

In [ ]:
# ----------------------------------------
# PLOT HISTORY
# ----------------------------------------

def plot_history(history):
    acc = history.history['binary_accuracy']
    val_acc = history.history['val_binary_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    pl.figure(figsize=(10, 5))
    pl.subplot(1, 2, 1)
    pl.plot(acc)
    pl.plot(val_acc)
    pl.ylabel('accuracy')
    pl.xlabel('epoch')
    pl.legend(['train', 'val'], loc='upper left')
    pl.grid()

    pl.subplot(1, 2, 2)
    pl.plot(loss)
    pl.plot(val_loss)
    pl.ylabel('loss')
    pl.xlabel('epoch')
    pl.legend(['train', 'val'], loc='upper left')
    pl.grid()

    pl.show();

plot_history(history)

## Saving the model

Save the model and its weights.

**Please don't forget to copy this vgg16_earthquake folder to your Google Drive for later use.**
To do so drag and drop this folder to drive/MyDrive/ through the Colab interface.

In [ ]:
model.save('vgg16_earthquakeoct2026.keras')